In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_joint_enc import NMF_logistic
from tensorflow import keras

In [ ]:
myDict = pickle.load(open('/home/austin/Aggression/Experiments/CombinedDatasets/Unbalanced_Elastic_12_enc_1.0.p',
                          'rb'))

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


In [ ]:
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_2020_3mice.mat',
                                               fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])

idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


mice_new = np.unique(mouse_new)
nMice = len(mice_new)


In [ ]:
A_enc = myDict['A_enc']
B_enc = myDict['B_enc']

In [ ]:
S = np.dot(X_new,A_enc) + B_enc

In [ ]:
id1 = mice_new[2] == mouse_new

In [ ]:
S_sub = S[id1]
y_sub = y_new[id1]

In [ ]:
S_sub.shape

In [ ]:
roc_auc_score(y_sub,S_sub[:,0])

In [ ]:
roc_auc_score(y_sub,S_sub[:,1])

In [ ]:
fpr,tpr,_ = roc_curve(y_sub,-1*S_sub[:,0])

In [ ]:
plt.plot(fpr,tpr)

In [ ]:
prec,recall,_ = precision_recall_curve(y_sub,-1*S_sub[:,0])

In [ ]:
plt.plot(recall,prec)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR Curve for Mouse 1830')
#plt.savefig('PRCurve.png')

In [ ]:
plt.plot(fpr,tpr-fpr)
plt.xlabel('False Positive Rate')
plt.ylabel('TPR-FPR')
plt.title('Decision Curve for Mouse 1830')
#plt.savefig('DecisionThreshold.png')

In [ ]:
plt.hist(S_sub[:,0])